# Defining a breakout

I will be using this notebook to find a way to accurately define what a breakout is for an NBA player so that I can find out what stats and meatrics I need to track in order to predict future breakouts.

### Threshold crossing

Analyze the dataset and define what a breakout seasons is.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent)) # project root

from utilities import load_data, features

df = load_data.load_clean_ppg()
df.shape

#### Games-played threshold

Instead of a fixed 40 games, a player needs the same *share* of their season's games (40/82). This keeps shortened seasons (1999, 2012, 2020, 2021) from being treated unfairly. The ppg change is computed on all rows first, then only back-to-back seasons where both years pass the threshold are kept, so skipped years don't create fake jumps.

In [ ]:
# Adds season_len, enough_games, ppg_change, season_gap and prev_enough (see utilities/features.py)
df = features.add_season_changes(df)

# Only compare back-to-back seasons where both seasons have enough games
usable = df[features.usable_mask(df)].copy()
usable.shape

In [5]:
# Distribution of ppg jumps: where would a breakout cutoff sit?
display(usable["ppg_change"].describe(percentiles=[.5, .75, .9, .95, .99]))

# The 20 biggest jumps: do these match seasons you'd call breakouts?
usable.sort_values("ppg_change", ascending=False)[["player", "season", "pts_per_game", "ppg_change"]].head(20)

count    11122.000000
mean        -0.014799
std          3.176809
min        -13.200000
50%         -0.100000
75%          1.900000
90%          4.000000
95%          5.400000
99%          8.300000
max         17.800000
Name: ppg_change, dtype: float64

,player,season,pts_per_game,ppg_change
16455,Dale Ellis,1987,24.9,17.8
15429,Tony Campbell,1990,23.2,17.0
16038,Kelly Tripucka,1989,22.6,15.1
15917,Reggie Lewis,1989,18.5,14.0
5075,CJ McCollum,2016,20.8,14.0
3132,Devonte' Graham,2020,18.2,13.5
15378,Orlando Woolridge,1991,25.1,12.4
17218,Derek Smith,1985,22.1,12.3
16105,Frank Brickowski,1988,16.0,12.0
1443,Cam Thomas,2024,22.5,11.9


In [ ]:
# Sanity checks
# 1) Thresholds should be lower in shortened seasons (1999, 2012, 2020, 2021)
print((features.MIN_SHARE * df["season_len"]).groupby(df["season"]).first().loc[[1998, 1999, 2011, 2012, 2020, 2021]].round(1))

# 2) Usable rows per season should look similar in neighboring years
print(usable.groupby("season").size().loc[[1998, 1999, 2000, 2011, 2012, 2020, 2021]])

# 3) Share of consecutive-season pairs that are real (season_gap == 1)
print((df["season_gap"] == 1).mean())

In [ ]:
usable["breakout"] = (usable["ppg_change"] >= features.BREAKOUT_THRESHOLD).astype(int)
usable["breakout"].describe()

### Decision

A breakout is a ppg jump of at least `BREAKOUT_THRESHOLD` (5) over the previous season, which is roughly the top 5-6% of jumps. Only back-to-back seasons where both years pass the games-played threshold count.

These rules live in `utilities/features.py`. To build the dataset the model uses, run `python -m utilities.features` from the project root, then continue in `02_model_and_importance.ipynb`.